[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ach0000/DualScoreMatching/blob/main/tp/TP_melange_gaussien.ipynb)

# Pourquoi le score ne suffit pas : score matching simple et dual sur un mélange de gaussiennes

On reprend la figure 1 de l'article : un mélange de deux gaussiennes centrées de variances différentes, en dimension $d = 1000$, dont on connaît l'énergie exacte. Un petit modèle d'énergie, entraîné par score matching simple (score d'espace seul) puis dual (scores d'espace et de temps), est comparé à la vérité.

**Matériel** : tourne sur un ordinateur personnel (CPU) en quelques minutes ; sur Colab, exécuter d'abord la cellule d'installation.

**Profils** : profil réduit (par défaut), `N_ITERATIONS = 4000` (environ 2 minutes par modèle, 6 minutes en tout sur CPU) ; profil GPU, 20 000 itérations comme les auteurs. Le notebook des auteurs correspondant, adapté pour Colab, est `auteurs/simple_example_colab.ipynb`.

Le profil réduit est utilisé par défaut, même avec un GPU (le calcul se fait alors sur le GPU) ; pour les réglages des auteurs, remplacer `if False:` par `if True:` dans la cellule de réglages (profil GPU, non vérifié).

**Reprise** : les calculs longs sont mis en cache dans `resultats/cache/` ; une exécution interrompue reprend là où elle s'était arrêtée. Supprimer ce dossier pour tout recalculer.

In [ ]:
# Installation (Colab) : à exécuter en premier ; en local, ne fait que rendre le code importable
GITHUB_USER = "ach0000"  # propriétaire du fork
REPO = "DualScoreMatching"
BRANCH = "main"
# seuls ces fichiers du dépôt sont téléchargés (motifs de git sparse-checkout)
FICHIERS = ['/*.py',
            '/networks/',
            '/tp/tp_outils.py']

import os, sys, subprocess
if "google.colab" in sys.modules:
    root = os.path.join("/content", REPO)
    if not os.path.isdir(os.path.join(root, ".git")):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
                        "https://github.com/%s/%s.git" % (GITHUB_USER, REPO), root], check=True)
    subprocess.run(["git", "-C", root, "sparse-checkout", "set", "--no-cone", *FICHIERS], check=True)
    subprocess.run(["git", "-C", root, "checkout", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'einops', 'PyWavelets'], check=True)
    os.chdir(os.path.join(root, "tp"))
else:
    racine = os.getcwd()
    while not os.path.exists(os.path.join(racine, "main.py")) and os.path.dirname(racine) != racine:
        racine = os.path.dirname(racine)
    os.chdir(os.path.join(racine, "tp"))
sys.path.insert(0, os.getcwd())
print("Répertoire courant :", os.getcwd())

## 0. Réglages

Le modèle d'énergie est minuscule (un perceptron qui ne voit que le niveau de bruit) : plus de 4 fils de calcul
ralentissent l'entraînement. `N_ITERATIONS` vaut 4000 dans le profil réduit (environ 2 minutes par modèle sur CPU) et 20 000 dans le profil GPU,
comme les auteurs.

In [ ]:
%matplotlib inline
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
import tp_outils as T
T.nommer_figures("TP_melange_gaussien")   # figures nommées dans resultats/figures/
torch.set_num_threads(4)

GRAINE = 0
# Profil : réduit par défaut, que le calcul se fasse sur CPU ou sur GPU (T.DEVICE suit le matériel connecté).
# Réglages des auteurs (profil GPU, non vérifié, GPU indispensable) : remplacer False par True ci-dessous.
GPU = False
if False:
    GPU = torch.cuda.is_available()
N_ITERATIONS = 20_000 if GPU else 4000
print("calcul sur", T.DEVICE, "-", N_ITERATIONS, "itérations")

## 1. Le mélange et son énergie exacte

$p = \tfrac12\,\mathcal N(0, \sigma_1^2 I_d) + \tfrac12\,\mathcal N(0, \sigma_2^2 I_d)$ avec $\sigma_1 = 1$,
$\sigma_2 = 4$, $d = 1000$. Bruitée avec une variance $t$, c'est le mélange des $\mathcal N(0, (\sigma_k^2 + t) I_d)$,
d'énergie exacte

$$U(y, t) = -\log \sum_{k=1}^2 \tfrac12\,(2\pi(\sigma_k^2+t))^{-d/2} \exp\Big(-\frac{\|y\|^2}{2(\sigma_k^2+t)}\Big),$$

qui ne dépend que du rayon $r = \|y\|$ (classe `T.MelangeGaussien`). Les log-probabilités sont exprimées en
décibels par dimension : $\frac{10}{d}\log_{10} p = -\frac{10\,U}{d \ln 10}$.

In [ ]:
M = T.MelangeGaussien(d=1000, sigmas=(1.0, 4.0), poids=(0.5, 0.5))
d = M.d
X = M.echantillons(100_000, graine=GRAINE)       # (N, d)
r = X.norm(dim=1)
dB = lambda e: T.nats_vers_dB(e, d)
r0 = math.sqrt(d) * torch.linspace(0.01, 5, 500)    # grille de rayons

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(r0 / math.sqrt(d), dB(M.energie(r0)), lw=2, label="log p exacte")
ax.set_xlabel("$\\|x\\|/\\sqrt{d}$"); ax.set_ylabel("log p (dB/dim)"); ax.set_ylim(-15, 0)
ax2 = ax.twinx()
ax2.hist((r / math.sqrt(d)).numpy(), bins=100, density=True, alpha=0.4, color="tab:blue")
ax2.set_yticks([]); ax.legend(loc="upper right"); ax.set_title("Mélange de deux gaussiennes, d = 1000")
T.montrer("energie_exacte")

for k, s in enumerate(M.sigmas.tolist()):
    print("composante %d : rayon typique %.2f sqrt(d), entropie d/2 log(2 pi e sigma^2) = %.2f dB/dim"
          % (k + 1, s, dB(torch.tensor(d / 2 * math.log(2 * math.pi * math.e * s ** 2))).item()))
print("entropie du mélange E[U(x)] = %.2f dB/dim" % dB(M.energie(r).mean()).item())
entre = ((r / math.sqrt(d) > 1.5) & (r / math.sqrt(d) < 3.5)).float().mean().item()
print("fraction des échantillons de rayon entre 1,5 et 3,5 sqrt(d) : %.2e" % entre)

## 2. Le paysage d'énergie dans l'espace et le temps (dimension 1)

Figure 1 (droite) de l'article : en dimension 1, un mélange de deux gaussiennes étroites, centrées en $\pm 1$,
de poids $0{,}3$ et $0{,}7$. Le chemin direct entre les modes à bruit très faible traverse une barrière d'énergie ;
un chemin qui passe par des niveaux de bruit plus élevés l'évite.

In [ ]:
mu, s2, w = np.array([-1.0, 1.0]), 0.05 ** 2, np.array([0.3, 0.7])

def U1(y, t):
    v = s2 + t
    dens = (w / np.sqrt(2 * np.pi * v) * np.exp(-(y[..., None] - mu) ** 2 / (2 * v))).sum(-1)
    return -np.log(dens + 1e-300)

ys = np.linspace(-2, 2, 400)
logts = np.linspace(-3, 1, 300)
carte = U1(ys[None, :], 10 ** logts[:, None, None])
# chemin 1 : à t = 1e-3 ; chemin 2 : demi-ellipse qui monte jusqu'à t = 1
u = np.linspace(0, np.pi, 400)
chemins = {"direct (t = 1e-3)": (np.linspace(-1, 1, 400), np.full(400, -3.0)),
           "par les grands bruits": (-np.cos(u), -3 + 3 * np.sin(u))}

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
im = axes[0].imshow(np.clip(carte, None, 20), origin="lower", aspect="auto", extent=[-2, 2, -3, 1], cmap="viridis")
plt.colorbar(im, ax=axes[0], label="U(y, t) (écrêtée à 20)")
for (nom, (yc, lt)), c in zip(chemins.items(), ["tab:green", "tab:red"]):
    axes[0].plot(yc, lt, color=c, lw=2, label=nom)
    axes[1].plot(np.linspace(0, 1, len(yc)), U1(yc, 10 ** lt[:, None])[:, 0] if False else
                 np.array([U1(np.array([a]), 10 ** b)[0] for a, b in zip(yc, lt)]), color=c, label=nom)
    print("%-25s énergie maximale le long du chemin : %.1f nats" % (nom, max(U1(np.array([a]), 10 ** b)[0]
                                                                         for a, b in zip(yc, lt))))
axes[0].set_xlabel("y"); axes[0].set_ylabel("log10 t"); axes[0].legend(fontsize=8)
axes[1].set_xlabel("abscisse le long du chemin"); axes[1].set_ylabel("U (nats)"); axes[1].legend(fontsize=8)
axes[1].set_title("Énergie le long des deux chemins")
T.montrer("paysage_espace_temps")

## 3. Entraîner un modèle d'énergie par score matching simple ou dual

Le modèle des auteurs (`T.EBMRadial`, équation (44)) : $U_\theta(y,t) = -\log\sum_{i=1}^2 e^{-a_i(t)\|y\|^2 - d\,b_i(t)}$,
où $(a_1,a_2,b_1,b_2)$ sont produits par un perceptron à partir de $\log(t + t_{\min})$. Cette famille contient
l'énergie exacte. Il est entraîné avec la classe `EnergyModel` des auteurs, qui calcule $\nabla_y U_\theta$ et
$\partial_t U_\theta$ par autograd, et avec la perte de `T.perte_dsm` (identique à `compute_metrics` de `main.py`) :

$$\underbrace{\frac1d\big\|\sqrt t\,\nabla_yU_\theta(y,t) - z\big\|^2}_{\text{terme d'espace}}
\;+\;\underbrace{\Big(\frac td\,\partial_tU_\theta(y,t) - \frac12\Big(1 - \frac{\|z\|^2}{d}\Big)\Big)^2}_{\text{terme de temps}},
\qquad y = x + \sqrt t\, z,\quad \log t \sim \mathcal U([\log 10^{-2}, \log 10^{2}]).$$

Score matching simple : terme d'espace seul. Dual : somme des deux. Dans les deux cas, la constante de normalisation
est fixée ensuite par l'équation (9) : $C = \frac d2\log(2\pi e\, t_{\max}) - \mathbb E_y[U_\theta(y, t_{\max})]$.

In [ ]:
modeles = {}
for dual in [False, True]:
    nom = "dual" if dual else "simple"
    print("score matching", nom)
    modeles[nom] = T.entrainer_ebm_radial(X, dual=dual, n_iterations=N_ITERATIONS, graine=GRAINE, cache="melange")
    print("  constante de normalisation C = %.1f nats" % modeles[nom][1])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for nom, (_, _, hist) in modeles.items():
    lisse = lambda v: np.convolve(v, np.ones(50) / 50, mode="valid")
    axes[0].semilogy(lisse(hist[:, 0]), label=nom); axes[1].semilogy(lisse(hist[:, 1]), label=nom)
axes[0].set_title("terme d'espace"); axes[1].set_title("terme de temps (non optimisé en simple)")
for ax in axes:
    ax.set_xlabel("itération"); ax.legend()
T.montrer("pertes")

## 4. Les énergies apprises

Comme les auteurs, on évalue $U_\theta(y, t_{\min})$ (plus petit bruit vu à l'entraînement) et on ajoute la
constante $C$. Pour le score matching simple, on montre aussi l'énergie « ajustée » : la constante est choisie pour
que l'énergie moyenne sur les données égale l'entropie exacte (meilleur recalage global possible).

In [ ]:
t_min = 1e-2
def energie_apprise(nom, rayons):
    reseau, C, _ = modeles[nom]
    with torch.no_grad():
        return reseau.forward_r(rayons, torch.full_like(rayons, t_min)) + C

vraie = M.energie(r0).float()
apprises = {nom: energie_apprise(nom, r0) for nom in modeles}
decalage = M.energie(r).mean().item() - energie_apprise("simple", r).mean().item()
apprises["simple (ajusté)"] = apprises["simple"] + decalage

styles = {"simple": ("tab:green", "--"), "simple (ajusté)": ("tab:green", ":"), "dual": ("tab:red", "--")}
x0 = (r0 / math.sqrt(d)).numpy()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(x0, dB(vraie), lw=3, label="exacte")
for nom, e in apprises.items():
    axes[0].plot(x0, dB(e), color=styles[nom][0], ls=styles[nom][1], lw=2, label=nom)
axes[0].set_ylim(-15, 0); axes[0].set_ylabel("log p (dB/dim)")
pas = (r0[1] - r0[0]) / math.sqrt(d)
derivee = lambda e: (dB(e)[1:] - dB(e)[:-1]) / pas
axes[1].plot((x0[1:] + x0[:-1]) / 2, derivee(vraie), lw=3, label="exacte")
for nom in ["simple", "dual"]:
    axes[1].plot((x0[1:] + x0[:-1]) / 2, derivee(apprises[nom]), color=styles[nom][0], ls="--", lw=2, label=nom)
axes[1].set_ylim(-8.5, 0.5); axes[1].set_ylabel("dérivée radiale de log p (dB/dim par unité)")
for ax in axes:
    ax.set_xlabel("$\\|x\\|/\\sqrt{d}$"); ax.legend(fontsize=8)
    ax2 = ax.twinx(); ax2.hist((r / math.sqrt(d)).numpy(), bins=100, density=True, alpha=0.3); ax2.set_yticks([])
T.montrer("energies_apprises")

print("%-18s %14s %14s %16s" % ("", "erreur mode 1", "erreur mode 2", "U(mode 2)-U(mode 1)"))
rayons_typiques = torch.tensor([1.0, 4.0]) * math.sqrt(d)
ev = M.energie(rayons_typiques).float()
print("%-18s %14s %14s %13.2f dB" % ("exacte", "", "", (dB(ev[1]) - dB(ev[0])).item()))
for nom in modeles:
    e = energie_apprise(nom, rayons_typiques)
    err = dB(e) - dB(ev)
    print("%-18s %11.2f dB %11.2f dB %13.2f dB" % (nom, err[0].item(), err[1].item(), (dB(e[1]) - dB(e[0])).item()))

## 5. Le score de temps appris

On compare $\frac td\,\partial_t U(y, t)$ appris et exact, au rayon typique de chaque composante bruitée,
$r = \sqrt{d(\sigma_k^2 + t)}$, pour $t$ entre $10^{-2}$ et $10^2$.

In [ ]:
ts = torch.logspace(-2, 2, 60)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for k, ax in enumerate(axes):
    rk = torch.sqrt(d * (M.sigmas[k].float() ** 2 + ts))
    ax.semilogx(ts, (ts * M.score_temps(rk, ts.double()).float() / d), lw=3, label="exact")
    for nom in modeles:
        reseau = modeles[nom][0]
        tt = ts.clone().requires_grad_(True)
        e = reseau.forward_r(rk, tt)
        g = torch.autograd.grad(e.sum(), tt)[0]
        ax.semilogx(ts, (ts * g / d).detach(), ls="--", color=styles[nom][0], lw=2, label=nom)
    ax.set_title("rayon typique de la composante %d" % (k + 1)); ax.set_xlabel("t"); ax.set_ylabel("t ∂U/∂t / d")
    ax.legend(fontsize=8)
T.montrer("score_temps")

## 6. Une régression estime une espérance conditionnelle

Les deux cibles de la perte, $(y - x)/t$ et $\frac d{2t} - \frac{\|y - x\|^2}{2t^2}$, ne dépendent pas du modèle :
leur espérance sachant $y$ est le score d'espace (Miyasawa--Tweedie) et le score de temps (équation (4)). On le
vérifie à $t = 1$ : on tire $(x, z)$, on regroupe les $y$ par tranches de rayon et l'on moyenne les cibles dans
chaque tranche.

In [ ]:
t = 1.0
Xs = M.echantillons(200_000, graine=GRAINE + 1)
torch.manual_seed(GRAINE)
Y = Xs + math.sqrt(t) * torch.randn_like(Xs)
rY = Y.norm(dim=1)
cible_espace = ((Y - Xs) * Y).sum(1) / (t * rY)                # composante radiale de (y - x)/t
cible_temps = d / (2 * t) - ((Y - Xs) ** 2).sum(1) / (2 * t ** 2)
bords = torch.quantile(rY, torch.linspace(0, 1, 41))
centres, moy_e, moy_t = [], [], []
for a, b in zip(bords[:-1], bords[1:]):
    sel = (rY >= a) & (rY < b)
    centres.append(rY[sel].mean()); moy_e.append(cible_espace[sel].mean()); moy_t.append(cible_temps[sel].mean())
centres = torch.stack(centres)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
grille = torch.linspace(rY.min(), rY.max(), 300)
axes[0].plot(grille / math.sqrt(d), M.score_radial(grille, t), label="score d'espace exact (radial)")
axes[0].plot(centres / math.sqrt(d), moy_e, "o", ms=3, label="moyenne de (y-x)/t par tranche")
axes[1].plot(grille / math.sqrt(d), M.score_temps(grille, t), label="score de temps exact")
axes[1].plot(centres / math.sqrt(d), moy_t, "o", ms=3, label="moyenne de la cible par tranche")
for ax in axes:
    ax.set_xlabel("$\\|y\\|/\\sqrt{d}$"); ax.legend(fontsize=8)
T.montrer("regression_cibles")
print("écart-type de la cible de temps : %.1f ; amplitude de son espérance : %.1f"
      % (cible_temps.std().item(), torch.stack(moy_t).abs().max().item()))

## 7. Le score matching simple dépend de l'initialisation

L'écart d'énergie entre les deux modes n'est pas contraint par le score matching simple : on recommence avec deux
autres graines.

In [ ]:
for graine in [GRAINE + 1, GRAINE + 2]:
    reseau, C, _ = T.entrainer_ebm_radial(X, dual=False, n_iterations=N_ITERATIONS, graine=graine, afficher_tous=0,
                                          cache="melange")
    with torch.no_grad():
        e = reseau.forward_r(rayons_typiques, torch.full_like(rayons_typiques, t_min)) + C
    err = dB(e) - dB(ev)
    print("graine %d : erreur mode 1 %.2f dB, mode 2 %.2f dB, U(mode 2)-U(mode 1) = %.2f dB (exact %.2f)"
          % (graine, err[0].item(), err[1].item(), (dB(e[1]) - dB(e[0])).item(), (dB(ev[1]) - dB(ev[0])).item()))